# AG News Text Classification — Production Training Pipeline
### MLOps Group Project | Member C: Model Sourcing, Training & Deployment

This notebook fine-tunes **DistilBERT** (`distilbert-base-uncased`) on the AG News 4-class news topic classification dataset using a **GPU T4** Kaggle environment, with:

-  **Three experimental versions** (`v1`, `v2`, `v3`) systematically evaluating variations in learning rate and epochs
-  **Live tracking** in Weights & Biases under project `mlops-ag_news_classification-distilbert`
-  **Comprehensive metrics**: loss, accuracy, precision, recall, F1 (weighted + macro), confusion matrix
-  **Automatic deployment** of the best model to the Hugging Face Hub (public profile)
-  **Full reproducibility** (deterministic seeds, pinned imports, FP16 mixed precision)

---

###  Experimental Matrix & Hyperparameters
To discover the optimal configuration for deployment, the training pipeline sequentially executes three different hyperparameter iterations:

| Version | Learning Rate | Batch Size | Epochs | Experimental Focus |
| :--- | :---: | :---: | :---: | :--- |
| **v1** | 2e-5 | 32 | 3 | Baseline configuration benchmark |
| **v2** | 2e-5 | 32 | 5 | Impact of extended training duration (epochs) |
| **v3** | 3e-5 | 32 | 3 | Impact of an increased learning rate |

---

###  Model Selection Rationale

**Chosen model:** [`distilbert-base-uncased`](https://huggingface.co/distilbert-base-uncased) — 67M parameters, ~265 MB checkpoint (~250 MB in FP16, **fits the < 200 MB target after FP16 conversion / quantization**).

**Why DistilBERT (per the official model card):**

| Criterion | DistilBERT vs BERT-base |
|-----------|------------------------|
| Parameters | 67M vs 110M (**40% smaller**) |
| Inference speed | **60% faster** |
| Accuracy retention | **~97% of BERT performance** (per Sanh et al., 2019) |
| GLUE score | 77.0 (vs 79.5 for BERT-base) |
| Training stability | Excellent on short-text classification (AG News headlines) |

**Alternatives considered & rejected:**
- `bert-base-uncased` — too large (~440 MB), exceeds size budget
- `albert-base-v2` — sharing weights makes fine-tuning less stable
- `tinybert` — accuracy drop too steep for a graded assignment
- `mobilebert` — slower CPU inference despite smaller size

DistilBERT gives us the best **accuracy / size / speed** trade-off for this task.

## Step 0 — Sync Member B's Data Prep Repository
Pulls the latest `data_prep_B` branch which contains `train.csv`, `test.csv`, `id2label.json`, and helper functions.

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/mlops"
REPO_URL = "https://github.com/pratikktiwari/mlops.git"
BRANCH_NAME = "data_prep_B"

if not os.path.exists(REPO_DIR):
    print(f" Cloning '{BRANCH_NAME}' branch into {REPO_DIR}...")
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL, REPO_DIR], check=True)
    print(" Repository cloned successfully.")
else:
    print(f" Repository exists. Pulling latest '{BRANCH_NAME}' updates...")
    cwd = os.getcwd()
    try:
        os.chdir(REPO_DIR)
        subprocess.run(["git", "fetch", "origin"], check=True, stdout=subprocess.DEVNULL)
        subprocess.run(["git", "checkout", BRANCH_NAME], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(["git", "pull", "origin", BRANCH_NAME], check=True)
        print(f" Repository synced to latest '{BRANCH_NAME}' commit.")
    except subprocess.CalledProcessError as e:
        print(f" Git operation issue: {e}")
    finally:
        os.chdir(cwd)

## Step 1 — Experiment Configuration

**Multi-Version Training Setup:** Three experiment versions (`v1`, `v2`, `v3`) are trained sequentially by calling `train_version()` with per-version hyperparameters. `VERSION_CONFIG` is removed — hyperparameters are passed directly as function arguments for clean, explicit control.

### Version hyperparameters:
| Version | Learning Rate | Batch Size | Epochs |
|---------|--------------|------------|--------|
| v1 | 2e-5 | 32 | 3 |
| v2 | 2e-5 | 32 | 5 |
| v3 | 3e-5 | 32 | 3 |

In [ ]:
# ==========================================
# STEP 1: EXPERIMENT CONFIGURATION
# ==========================================

# --- Model & Tracking ---
MODEL_NAME      = 'distilbert-base-uncased'
WANDB_ENTITY    = 'g25ait2133-indian-institute-technology-jodhpur'   # Your W&B team/entity
WANDB_PROJECT   = 'mlops-ag_news_classification-distilbert'          # Required project name
HF_USERNAME     = 'mehtayash12345678'                                # Your HF profile
HF_MODEL_NAME   = 'mlops-ag_news_classification-distilbert'          # Repo under your profile
HF_MODEL_REPO   = f'{HF_USERNAME}/{HF_MODEL_NAME}'                   # → mehtayash12345678/mlops-ag_news_classification-distilbert

# --- Shared / held-constant hyperparameters ---
WEIGHT_DECAY        = 0.01
MAX_LENGTH          = 128
WARMUP_RATIO        = 0.1
TRAIN_SUBSET        = 24000        # 6,000 per class, stratified
LOGGING_STEPS       = 50           # Smooth W&B curves
SEED                = 42

# --- Deployment toggle ---
# Set to True on your BEST run (after comparing versions in W&B)
PUSH_TO_HUB = True

print(f' W&B:  https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}')
print(f' HF:   https://huggingface.co/{HF_MODEL_REPO}')
print('\n Configuration loaded — VERSION_CONFIG removed; hyperparameters passed per-call into train_version()')


## Step 2 — Install Dependencies & Set Deterministic Seeds
Pins critical packages and locks all random sources for reproducibility (Python `random`, NumPy, PyTorch CPU, PyTorch CUDA).

In [ ]:
# ==========================================
# STEP 2: DEPENDENCIES & REPRODUCIBILITY
# ==========================================
!pip install -q --no-warn-conflicts transformers datasets evaluate wandb huggingface_hub scikit-learn

import os
import random
import numpy as np
import pandas as pd
import torch

# Lock all random sources for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Environment diagnostics
print("--- Environment ---")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"GPU count:      {torch.cuda.device_count()}")
    print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3 — Load Kaggle Secrets & Authenticate
Securely retrieves `WANDB_API_KEY` and `HF_TOKEN` from Kaggle Secrets vault (Add-ons → Secrets). Never hard-code tokens.

In [ ]:
# ==========================================
# STEP 3: SECURE CREDENTIALS
# ==========================================
# IMPORTANT: Set WANDB_MODE BEFORE importing wandb to prevent auto-init warnings
os.environ['WANDB_MODE'] = 'disabled'

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login
import wandb

secrets = UserSecretsClient()
WANDB_API_KEY = None
HF_TOKEN      = None

# --- Fetch secrets ---
try:
    WANDB_API_KEY = secrets.get_secret('WANDB_API_KEY')
    print(" WANDB_API_KEY retrieved")
except Exception as e:
    print(f"  WANDB_API_KEY missing: {e}")

try:
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print(" HF_TOKEN retrieved")
except Exception as e:
    print(f"  HF_TOKEN missing: {e}")

# --- Hugging Face login ---
if HF_TOKEN:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print(" Hugging Face authenticated")
else:
    print(" HF login skipped — model push will be disabled")
    PUSH_TO_HUB = False

# --- W&B login (online or offline fallback) ---
if WANDB_API_KEY:
    try:
        os.environ['WANDB_API_KEY'] = WANDB_API_KEY
        os.environ['WANDB_MODE']    = 'online'
        wandb.login(key=WANDB_API_KEY)
        print(" W&B authenticated — ONLINE mode")
    except Exception as e:
        print(f" W&B online auth failed: {e} — falling back to OFFLINE mode")
        os.environ['WANDB_MODE'] = 'offline'
else:
    os.environ['WANDB_MODE'] = 'offline'
    print("  Running W&B in OFFLINE mode (no API key)")

print(f"\n Status: W&B={os.environ['WANDB_MODE'].upper()} | HF={'OK' if HF_TOKEN else 'SKIPPED'}")

## Step 4 — Load Prepared Data & Label Mapping
Loads Member B's preprocessed `train.csv`, `test.csv`, and `id2label.json`. The label mapping ensures the model's output head aligns with the project-wide schema.

In [ ]:
# ==========================================
# STEP 4: DATA LOADING & LABEL MAPPING
# ==========================================
import json

PREP_DIR   = '/kaggle/working/mlops'
train_path = os.path.join(PREP_DIR, 'train.csv')
test_path  = os.path.join(PREP_DIR, 'test.csv')
json_path  = os.path.join(PREP_DIR, 'id2label.json')

# Verify files exist
for p in [train_path, test_path, json_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required file missing: {p}. Re-run Step 0.")

# --- Load CSVs ---
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# --- Load and standardize label mapping (keys must be int for HF Trainer) ---
with open(json_path, 'r') as f:
    id2label_raw = json.load(f)

ID2LABEL   = {int(k): v for k, v in id2label_raw.items()}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

# --- Diagnostics ---
print("--- Dataset Diagnostics ---")
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Classes ({NUM_LABELS}): {ID2LABEL}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"\nClass balance (train):\n{train_df['label'].value_counts().sort_index()}")
print(f"\nSample row:\n{train_df.iloc[0].to_dict()}")

## Step 5 — Stratified Subsample
**Important:** Subsample **before** tokenization to save ~75% compute time. Stratification ensures each class remains balanced (6,000 samples per class).

In [ ]:
# ==========================================
# STEP 5: STRATIFIED SUBSAMPLE (PRE-TOKENIZATION)
# ==========================================
from sklearn.model_selection import train_test_split

if TRAIN_SUBSET and len(train_df) > TRAIN_SUBSET:
    train_df, _ = train_test_split(
        train_df,
        train_size=TRAIN_SUBSET,
        stratify=train_df['label'],
        random_state=SEED,
    )
    train_df = train_df.reset_index(drop=True)

print(f" Train rows after subsample: {len(train_df):,}")
print(f"Per-class distribution:\n{train_df['label'].value_counts().sort_index()}")

## Step 6 — Load Tokenizer & Model
Loads DistilBERT's tokenizer and instantiates the sequence-classification head with our `ID2LABEL` / `LABEL2ID` mappings. This ensures the model's config matches Member B's schema (essential for downstream inference).

In [ ]:
# ==========================================
# STEP 6: TOKENIZER & MODEL
# ==========================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Model with custom classification head ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f" Tokenizer loaded:  {MODEL_NAME}")
print(f" Model loaded with {NUM_LABELS} output classes")
print(f"   Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"   Config id2label:  {model.config.id2label}")

## Step 7 — Tokenize Datasets
Converts text into token IDs using the DistilBERT tokenizer, with truncation at `MAX_LENGTH=128`. Padding is handled dynamically per-batch by the `DataCollatorWithPadding` for efficiency.

In [ ]:
# ==========================================
# STEP 7: TOKENIZATION
# ==========================================
def to_hf(df):
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

train_ds = to_hf(train_df)
test_ds  = to_hf(test_df)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True, desc="Tokenizing train")
test_ds  = test_ds.map(tokenize_fn,  batched=True, desc="Tokenizing test")

# HF Trainer expects column named 'labels'
train_ds = train_ds.rename_column('label', 'labels')
test_ds  = test_ds.rename_column('label', 'labels')

print(f" Train tokenized: {len(train_ds):,} examples")
print(f" Test tokenized:  {len(test_ds):,} examples")
print(f"\nFeatures: {train_ds.features}")

## Step 8 — `train_version()` Function Definition

`VERSION_CONFIG` has been removed from the global constants. Instead, each experiment's hyperparameters (`learning_rate`, `BATCH_SIZE`, `EPOCHS`, `run_version`) are passed **directly as arguments** to `train_version()`. The function encapsulates the full pipeline: W&B init → Training → Evaluation → Confusion matrix → HF deployment → Summary.

Call it once per version:
```python
train_version(run_version='v1', learning_rate=2e-5, BATCH_SIZE=32, EPOCHS=3)
train_version(run_version='v2', learning_rate=2e-5, BATCH_SIZE=32, EPOCHS=5)
train_version(run_version='v3', learning_rate=3e-5, BATCH_SIZE=32, EPOCHS=3)
```

In [ ]:
# ==========================================
# TRAIN_VERSION FUNCTION
# Encapsulates Steps 8-13 for a single experimental run.
# Hyperparameters are passed as arguments — no VERSION_CONFIG dict needed.
# ==========================================

import json
from transformers import (
    TrainingArguments, Trainer, DataCollatorWithPadding,
    AutoTokenizer, AutoModelForSequenceClassification,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report,
)
from huggingface_hub import upload_file
import wandb
import torch

def train_version(
    run_version: str,
    learning_rate: float,
    BATCH_SIZE: int,
    EPOCHS: int,
):
    """
    Fine-tune DistilBERT for one experiment version.

    Parameters
    ----------
    run_version   : str    — version label, e.g. 'v1', 'v2', 'v3'
    learning_rate : float  — AdamW learning rate (e.g. 2e-5, 3e-5)
    BATCH_SIZE    : int    — per-device train batch size (e.g. 32)
    EPOCHS        : int    — number of training epochs (e.g. 3 or 5)
    """

    # Derive a descriptive run name from the arguments
    run_name = f'distilbert_{run_version}'

    print('=' * 70)
    print(f'  Starting Experiment: {run_version.upper()}')
    print(f'    learning_rate = {learning_rate}')
    print(f'    BATCH_SIZE    = {BATCH_SIZE}')
    print(f'    EPOCHS        = {EPOCHS}')
    print(f'    run_name      = {run_name}')
    print('=' * 70)

    # ── 8A: Reload tokenizer & model for a fresh run ──────────────────────
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    _model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    # ── 8B: W&B run initialization ─────────────────────────────────────────
    if wandb.run is not None:
        wandb.finish()

    wandb_config = {
        'model':          MODEL_NAME,
        'version':        run_version,
        'epochs':         EPOCHS,
        'batch_size':     BATCH_SIZE,
        'learning_rate':  learning_rate,
        'weight_decay':   WEIGHT_DECAY,
        'warmup_ratio':   WARMUP_RATIO,
        'max_length':     MAX_LENGTH,
        'train_samples':  len(train_df),
        'test_samples':   len(test_df),
        'num_labels':     NUM_LABELS,
        'fp16':           True,
        'platform':       'Kaggle T4 GPU',
        'dataset':        'AG News (4-class)',
        'seed':           SEED,
    }

    run = wandb.init(
        entity  = WANDB_ENTITY,
        project = WANDB_PROJECT,
        name    = run_name,
        tags    = ['distilbert', 'ag-news', f'lr-{learning_rate}', run_version],
        config  = wandb_config,
        reinit  = True,
    )
    print(f' W&B run: {run.url if run else "offline"}')

    # ── 8C: Dynamic padding collator ──────────────────────────────────────
    data_collator = DataCollatorWithPadding(tokenizer=_tokenizer)

    # ── 8D: Metrics ───────────────────────────────────────────────────────
    def compute_metrics(pred):
        labels = pred.label_ids
        preds  = pred.predictions.argmax(-1)
        return {
            'accuracy':    accuracy_score(labels, preds),
            'f1_weighted': f1_score(labels, preds, average='weighted'),
            'f1_macro':    f1_score(labels, preds, average='macro'),
            'precision':   precision_score(labels, preds, average='weighted', zero_division=0),
            'recall':      recall_score(labels, preds, average='weighted', zero_division=0),
        }

    # ── 8E: Training arguments ─────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir                  = f'./results-{run_version}',
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        learning_rate               = learning_rate,
        weight_decay                = WEIGHT_DECAY,
        warmup_ratio                = WARMUP_RATIO,
        fp16                        = torch.cuda.is_available(),
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        logging_strategy            = 'steps',
        logging_steps               = LOGGING_STEPS,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1_weighted',
        greater_is_better           = True,
        report_to                   = 'wandb',
        run_name                    = run_name,
        seed                        = SEED,
        dataloader_num_workers      = 2,
        disable_tqdm                = False,
    )

    # ── 8F: Trainer ───────────────────────────────────────────────────────
    trainer = Trainer(
        model            = _model,
        args             = training_args,
        train_dataset    = train_ds,
        eval_dataset     = test_ds,
        processing_class = _tokenizer,
        data_collator    = data_collator,
        compute_metrics  = compute_metrics,
    )

    # ── 8G: Training ──────────────────────────────────────────────────────
    print(f'\n  Training {run_version.upper()} (LR={learning_rate}, BS={BATCH_SIZE}, EP={EPOCHS})')
    print(f'   Watch live: {run.url if run else "offline"}\n')

    train_result    = trainer.train()
    train_metrics   = train_result.metrics
    trainer.log_metrics('train', train_metrics)
    trainer.save_metrics('train', train_metrics)
    print(f'\n Training complete!')
    print(f'   Final train loss: {train_metrics.get("train_loss", "n/a"):.4f}')
    print(f'   Total runtime:    {train_metrics.get("train_runtime", 0):.1f}s')

    # ── 8H: Evaluation + Confusion Matrix ─────────────────────────────────
    eval_metrics = trainer.evaluate()
    print('\n Final test metrics:')
    for k, v in eval_metrics.items():
        if isinstance(v, float):
            print(f'   {k:30s} = {v:.4f}')

    for k, v in eval_metrics.items():
        if isinstance(v, (int, float)):
            wandb.run.summary[f'final_{k}'] = v

    preds_output = trainer.predict(test_ds)
    y_pred       = preds_output.predictions.argmax(-1)
    y_true       = preds_output.label_ids
    class_names  = [ID2LABEL[i] for i in range(NUM_LABELS)]

    report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print('\n--- Classification Report ---')
    print(report_str)

    wandb.log({
        'confusion_matrix': wandb.plot.confusion_matrix(
            probs       = None,
            y_true      = y_true.tolist(),
            preds       = y_pred.tolist(),
            class_names = class_names,
        )
    })

    report_dict  = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    report_rows  = [[cls, report_dict[cls]['precision'], report_dict[cls]['recall'],
                     report_dict[cls]['f1-score'], report_dict[cls]['support']]
                    for cls in class_names]
    report_table = wandb.Table(columns=['class', 'precision', 'recall', 'f1', 'support'],
                               data=report_rows)
    wandb.log({'per_class_report': report_table})
    print('\n Confusion matrix + per-class report logged to W&B')

    # ── 8I: Deploy best model to Hugging Face Hub ─────────────────────────
    hf_url = None
    if PUSH_TO_HUB and HF_TOKEN:
        print(f'\n Pushing model to: https://huggingface.co/{HF_MODEL_REPO}')

        model_card = f"""---
language: en
license: apache-2.0
tags:
  - text-classification
  - distilbert
  - ag-news
  - mlops
datasets:
  - ag_news
metrics:
  - accuracy
  - f1
base_model: {MODEL_NAME}
---

# DistilBERT fine-tuned on AG News

This model is a fine-tuned version of [{MODEL_NAME}](https://huggingface.co/{MODEL_NAME}) on the AG News 4-class news topic classification dataset.

## Performance ({run_version.upper()})

| Metric         | Score |
|----------------|-------|
| Accuracy       | {eval_metrics.get('eval_accuracy', 0):.4f} |
| F1 (weighted)  | {eval_metrics.get('eval_f1_weighted', 0):.4f} |
| F1 (macro)     | {eval_metrics.get('eval_f1_macro', 0):.4f} |
| Precision      | {eval_metrics.get('eval_precision', 0):.4f} |
| Recall         | {eval_metrics.get('eval_recall', 0):.4f} |

## Hyperparameters

- Learning rate: {learning_rate}
- Batch size: {BATCH_SIZE}
- Epochs: {EPOCHS}
- Weight decay: {WEIGHT_DECAY}
- Warmup ratio: {WARMUP_RATIO}
- Max length: {MAX_LENGTH}
- Mixed precision: FP16
- Seed: {SEED}
"""

        import os
        os.makedirs(f'./results-{run_version}', exist_ok=True)
        with open(f'./results-{run_version}/README.md', 'w') as f:
            f.write(model_card)

        trainer.model.push_to_hub(HF_MODEL_REPO, commit_message=f'Upload {run_name}')
        _tokenizer.push_to_hub(HF_MODEL_REPO, commit_message='Upload tokenizer')
        upload_file(
            path_or_fileobj = f'./results-{run_version}/README.md',
            path_in_repo    = 'README.md',
            repo_id         = HF_MODEL_REPO,
            commit_message  = 'Add model card',
        )

        hf_url = f'https://huggingface.co/{HF_MODEL_REPO}'
        wandb.run.summary['huggingface_model_url']  = hf_url
        wandb.run.summary['huggingface_model_repo'] = HF_MODEL_REPO
        print(f'\n Model deployed → {hf_url}')
    else:
        print('  Skipped HF push (PUSH_TO_HUB=False or HF_TOKEN missing)')

    # ── 8J: Final summary ─────────────────────────────────────────────────
    wandb_run_url     = run.url if run else 'offline (not synced)'
    wandb_project_url = f'https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}'

    print('\n' + '=' * 70)
    print(f'                 {run_version.upper()} COMPLETE  ')
    print('=' * 70)
    print(f'\n Experiment : {run_version.upper()} ({run_name})')
    print(f'   LR={learning_rate}  BS={BATCH_SIZE}  EP={EPOCHS}')
    print(f'\n--- Final Metrics ---')
    print(f'   Accuracy:      {eval_metrics.get("eval_accuracy", 0):.4f}')
    print(f'   F1 (weighted): {eval_metrics.get("eval_f1_weighted", 0):.4f}')
    print(f'   F1 (macro):    {eval_metrics.get("eval_f1_macro", 0):.4f}')
    print(f'   Precision:     {eval_metrics.get("eval_precision", 0):.4f}')
    print(f'   Recall:        {eval_metrics.get("eval_recall", 0):.4f}')
    print(f'   Eval loss:     {eval_metrics.get("eval_loss", 0):.4f}')
    print(f'\n--- Deliverable URLs ---')
    print(f'    W&B Run:  {wandb_run_url}')
    print(f'    W&B Proj: {wandb_project_url}')
    if hf_url:
        print(f'    HF Model: {hf_url}')
    print('=' * 70 + '\n')

    wandb.finish()
    return eval_metrics


print(' train_version() function defined — ready to run v1, v2, v3')


## Step 9 — Run All Three Experiments

Each call to `train_version()` passes the hyperparameters for that version **directly as function arguments**. No global `VERSION_CONFIG` dictionary is used.

| Call | `run_version` | `learning_rate` | `BATCH_SIZE` | `EPOCHS` |
|------|--------------|-----------------|-------------|----------|
| 1st  | `'v1'`       | `2e-5`          | `32`        | `3`      |
| 2nd  | `'v2'`       | `2e-5`          | `32`        | `5`      |
| 3rd  | `'v3'`       | `3e-5`          | `32`        | `3`      |

### Direct links
- **W&B Project:** https://wandb.ai/g25ait2133-indian-institute-technology-jodhpur/mlops-ag_news_classification-distilbert
- **HF Model:** https://huggingface.co/mehtayash12345678/mlops-ag_news_classification-distilbert


In [ ]:
# ==========================================
# STEP 9: RUN ALL THREE EXPERIMENT VERSIONS
# Hyperparameters are passed directly — VERSION_CONFIG is NOT used.
# ==========================================

# ── v1: learning_rate=2e-5, BATCH_SIZE=32, EPOCHS=3 ──────────────────────
results_v1 = train_version(
    run_version   = 'v1',
    learning_rate = 2e-5,
    BATCH_SIZE    = 32,
    EPOCHS        = 3,
)

# ── v2: learning_rate=2e-5, BATCH_SIZE=32, EPOCHS=5 ──────────────────────
results_v2 = train_version(
    run_version   = 'v2',
    learning_rate = 2e-5,
    BATCH_SIZE    = 32,
    EPOCHS        = 5,
)

# ── v3: learning_rate=3e-5, BATCH_SIZE=32, EPOCHS=3 ──────────────────────
results_v3 = train_version(
    run_version   = 'v3',
    learning_rate = 3e-5,
    BATCH_SIZE    = 32,
    EPOCHS        = 3,
)

print('\n All three versions complete!')
print(f'   v1 F1: {results_v1.get("eval_f1_weighted", 0):.4f}')
print(f'   v2 F1: {results_v2.get("eval_f1_weighted", 0):.4f}')
print(f'   v3 F1: {results_v3.get("eval_f1_weighted", 0):.4f}')
